In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os

In [2]:
import infercnvpy as cnv

/home/augusta/anaconda3/envs/insitucnv_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Compare CNV clusters to histopathological annotations (or *ROI_status*)

## Read in dataset

In [ ]:
adata = sc.read(os.path.expanduser("~/storage3/augusta/insituCNV/InSituCNV/Colorectal_cancer_CosMxWTx/01_insituCNV/data/CosMx_221_common_CNVinf.h5ad"))

In [ ]:
adata

## CNV clusters vs ROI status

In [5]:
# Filter out the 'non-epi' cluster
adata = adata[adata.obs['c'] != 'non-epi']

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# CNV cluster colors
cnv_leiden_colors = {
    '0': '#FFCD4E',  # Cluster 0
    '1': '#677DB7',  # Cluster 1
    '2': '#FF773D',  # Cluster 2
    '3': '#9B1D20',  # Cluster 3
    'non-epi': '#cccccc',
}


# Create contingency table (counts of CNV clusters per ROI_status)
ct = pd.crosstab(adata.obs['ROI_status'], adata.obs['epi_cnv_leiden_res0.1'].astype(str))  # Ensure clusters are treated as strings

# Normalize within each ROI status
ct_norm = ct.div(ct.sum(axis=1), axis=0) * 100

# Get CNV Leiden cluster labels in the same order as in the contingency table
cnv_leiden_labels = ct_norm.columns.tolist()

# Plot as a stacked bar plot with correct colors from the dictionary
ct_norm.plot(
    kind="bar",
    stacked=True,
    color=[cnv_leiden_colors[str(cluster)] for cluster in cnv_leiden_labels],  # Access colors from the dictionary using string keys
    figsize=(4, 6),
    width=0.8
)

# Labels and title
plt.ylabel("Proportion (%)", fontsize=16)
plt.xlabel("ROI Status", fontsize=16)
plt.title("Proportion of CNV Leiden Clusters per ROI Status", fontsize=18)
plt.legend(title="CNV Leiden Cluster", bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=12)
plt.xticks(rotation=90, fontsize=14)
plt.yticks(fontsize=14)
plt.savefig('prop_of_clones_per_ROI_status.pdf')
plt.show()